### Problem Statement:

The objective of this project is to analyze product-level data from an e-commerce platform to understand category-wise assortment, pricing strategies, discount patterns, and customer engagement through ratings. The analysis focuses on identifying gaps such as low rating coverage, variations in discounting across categories, and differences in pricing behavior. The goal is to uncover actionable insights that can improve product discovery, optimize pricing and discount strategies, and enhance overall customer experience and conversion.

In [1]:
# Importing pandas
import pandas as pd
pd.set_option('display.max_rows', None)

In [2]:
# Loading the dataset
df = pd.read_csv('bigbasket_products.csv')

In [3]:
df.head(2)

,index,product,category,sub_category,brand,sale_price,market_price,type,rating,description
0,1,Garlic Oil - Vegetarian Capsule 500 mg,Beauty & Hygiene,Hair Care,Sri Sri Ayurveda,220.0,220.0,Hair Oil & Serum,4.1,This Product contains Garlic Oil that is known...
1,2,Water Bottle - Orange,"Kitchen, Garden & Pets",Storage & Accessories,Mastercook,180.0,180.0,Water & Fridge Bottles,2.3,"Each product is microwave safe (without lid), ..."


In [4]:
# To check the shape of dataframe
df.shape

(27555, 10)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27555 entries, 0 to 27554
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   index         27555 non-null  int64  
 1   product       27554 non-null  object 
 2   category      27555 non-null  object 
 3   sub_category  27555 non-null  object 
 4   brand         27554 non-null  object 
 5   sale_price    27555 non-null  float64
 6   market_price  27555 non-null  float64
 7   type          27555 non-null  object 
 8   rating        18929 non-null  float64
 9   description   27440 non-null  object 
dtypes: float64(3), int64(1), object(6)
memory usage: 2.1+ MB


In [6]:
# Null in Dataframe
df.isnull().sum()

index              0
product            1
category           0
sub_category       0
brand              1
sale_price         0
market_price       0
type               0
rating          8626
description      115
dtype: int64

In [7]:
# Duplicates in Dataframe
df.duplicated().sum()

0

In [8]:
# Removed 'description' column since it is not required for analysis
df = df.drop(columns=['description'])

In [9]:
# Removed rows with missing values in 'product' and 'brand' columns
df = df.dropna(subset=['product','brand'])

In [10]:
df['brand'] = df['brand'].astype(str).str.strip().str.lower()

In [11]:
df['product'] = df['product'].astype(str).str.strip().str.lower()

In [13]:
df['rating_status'] = df['rating'].apply(lambda x: 'Rated' if pd.notnull(x) else 'Not Rated')

In [14]:
#How many products are rated vs not rated?
df['rating_status'].value_counts()

Rated        18928
Not Rated     8625
Name: rating_status, dtype: int64

##### Insight:
1. Out of total products, 18,928 have ratings while 8,625 are unrated
2. This indicates that a significant portion of products (30-31%) lack customer feedback
3. While most products have ratings, a considerable gap still exists in user engagement

In [15]:
# Rechecking missing values after data cleaning
df.isnull().sum()

index               0
product             0
category            0
sub_category        0
brand               0
sale_price          0
market_price        0
type                0
rating           8625
rating_status       0
dtype: int64

In [16]:
df.shape

(27553, 10)

### Exploratory Data Analysis (EDA) – Product Assortment, Pricing, and Brand Insights

In [17]:
# Total SKU Count on the Platform
df['product'].nunique()

23510

##### Insight:
The platform offers 23,510 unique products (SKUs)

In [18]:
#How many unique categories exist?

total_categories = df.category.unique()

for i, category in enumerate(total_categories, start=1):
    print(f"{i}. {category}")

1. Beauty & Hygiene
2. Kitchen, Garden & Pets
3. Cleaning & Household
4. Gourmet & World Food
5. Foodgrains, Oil & Masala
6. Snacks & Branded Foods
7. Beverages
8. Bakery, Cakes & Dairy
9. Baby Care
10. Fruits & Vegetables
11. Eggs, Meat & Fish


##### Insight:
The platform offers products across 11 distinct categories

In [19]:
# How many unique subcategories exist in each category?

total_subcategories = df.groupby('category')['sub_category'].nunique().sort_values(ascending=False)\
                      .reset_index(name='no_of_subcategories')
total_subcategories

,category,no_of_subcategories
0,Gourmet & World Food,14
1,Snacks & Branded Foods,12
2,Beauty & Hygiene,10
3,Cleaning & Household,10
4,"Kitchen, Garden & Pets",10
5,"Foodgrains, Oil & Masala",9
6,"Bakery, Cakes & Dairy",8
7,Baby Care,7
8,Fruits & Vegetables,7
9,Beverages,6


##### Insight:
1. The number of subcategories varies across categories, indicating differences in product depth  
2. 'Gourmet & World Food' and 'Snacks & Branded Foods' have the highest subcategory diversity  
3. Categories like 'Beverages' and 'Eggs, Meat & Fish' have fewer subcategories, suggesting a more focused product range

In [20]:
#Which category have the highest SKU count?

sku_cnt = df.groupby('category')['product']\
            .nunique() \
            .sort_values(ascending=False).reset_index(name='sku_count')
sku_cnt

,category,sku_count
0,Beauty & Hygiene,6836
1,Gourmet & World Food,4101
2,"Kitchen, Garden & Pets",3181
3,Snacks & Branded Foods,2451
4,Cleaning & Household,2407
5,"Foodgrains, Oil & Masala",1996
6,Beverages,754
7,"Bakery, Cakes & Dairy",751
8,Baby Care,549
9,Fruits & Vegetables,353


##### Insight:
1. 'Beauty & Hygiene' has the highest SKU count, followed by 'Gourmet & World Food' and 'Kitchen, Garden & Pets'  
2. Indicates that these categories have the widest product assortment on the platform  
3. Categories like 'Eggs, Meat & Fish' and 'Fruits & Vegetables' have relatively fewer SKUs, suggesting a more limited or standardized product range

In [21]:
#Which categories have the highest average rating?

avg_rating = df.groupby('category')['rating']\
            .mean().sort_values(ascending=False)\
            .reset_index(name='average_rating')
avg_rating

,category,average_rating
0,Beverages,4.084676
1,"Foodgrains, Oil & Masala",4.062198
2,Baby Care,4.023790
3,Gourmet & World Food,3.984410
4,Snacks & Branded Foods,3.983313
5,Cleaning & Household,3.956667
6,Beauty & Hygiene,3.930655
7,"Bakery, Cakes & Dairy",3.911128
8,"Kitchen, Garden & Pets",3.734715
9,"Eggs, Meat & Fish",NaN


##### Insight:
1. After correcting the handling of missing ratings, the average ratings across categories increased significantly
2. Earlier results were biased due to replacing missing values with 0, which incorrectly lowered category averages
3. Most categories now show average ratings between 3.7 and 4.1, indicating generally positive customer feedback
4. Categories like 'Eggs, Meat & Fish' and 'Fruits & Vegetables' have no ratings available, hence appear as NaN
5. This highlights the importance of proper missing value treatment in rating-based analysis

In [22]:
# What is the average sale price and market price across categories?

avg_pricing = df.groupby('category')[['sale_price', 'market_price']].mean().rename(columns={
                                                'sale_price' : 'avg_sale_price', 
                                                'market_price' : 'avg_market_price'}).reset_index()
avg_pricing

,category,avg_sale_price,avg_market_price
0,Baby Care,534.946180,596.754098
1,"Bakery, Cakes & Dairy",142.802750,157.881316
2,Beauty & Hygiene,418.679197,493.535302
3,Beverages,239.803925,272.270362
4,Cleaning & Household,226.239001,262.191604
5,"Eggs, Meat & Fish",288.897486,325.835486
6,"Foodgrains, Oil & Masala",193.167500,230.131913
7,Fruits & Vegetables,50.889336,64.433662
8,Gourmet & World Food,319.854011,358.420885
9,"Kitchen, Garden & Pets",507.524615,659.657654


##### Insight:
1. Significant variation exists in pricing across categories
2. 'Baby Care' and 'Kitchen, Garden & Pets' have the highest average prices, indicating premium or high-value products
3. 'Fruits & Vegetables' and 'Snacks & Branded Foods' have lower average prices, reflecting everyday essential items
4. In all categories, average market price is higher than sale price, indicating consistent discounting across the platform

In [23]:
# How many distinct brands exist in each category?
df.groupby('category')['brand'].nunique().sort_values(ascending=False).reset_index(name='distinct_brand_cnt')

,category,distinct_brand_cnt
0,Beauty & Hygiene,644
1,Gourmet & World Food,545
2,Cleaning & Household,391
3,Snacks & Branded Foods,354
4,"Kitchen, Garden & Pets",240
5,"Foodgrains, Oil & Masala",235
6,Beverages,171
7,"Bakery, Cakes & Dairy",85
8,Baby Care,65
9,"Eggs, Meat & Fish",44


##### Insight:
1. Categories such as 'Beauty & Hygiene' (644 brands) and 'Gourmet & World Food' (546 brands) exhibit high brand diversity, indicating competitive and fragmented markets
2. In contrast, 'Fruits & Vegetables' has only 5 brands, suggesting dominance of unbranded or locally sourced products
3. This highlights structural differences between packaged goods categories and fresh produce categories

In [24]:
#How many distinct brands are on the platform?

df['brand'].nunique()

2312

##### Insight:
1. A total of 2,313 unique brands are present on the platform
2. This reflects a highly diverse and competitive product ecosystem
3. High brand diversity enhances customer choice but may also increase decision complexity

In [25]:
#Which Top 5 brands have the largest SKU presence?

df.groupby('brand')['product'].nunique().sort_values(ascending=False).head(5).reset_index(name='sku_count')

,brand,sku_count
0,fresho,438
1,bb royal,415
2,bb home,389
3,dp,240
4,fresho signature,170


##### Insight:
1. 'Fresho', 'bb Royal', and 'BB Home' have the highest SKU counts on the platform
2. These brands dominate the product catalog in terms of variety
3. Indicates strong presence of a few leading brands across multiple categories

In [26]:
#top 5 brands with highest average price

df.groupby('brand')['sale_price'].mean()\
.sort_values(ascending=False).reset_index(name='avg_sale_price').head(5)

,brand,avg_sale_price
0,farmina,10090.0
1,carolina herrera,6660.0
2,fitwhey,5900.0
3,paco rabanne,5800.0
4,bentley,5346.0


##### Insight:
1. Brands like 'Farmina', 'Carolina Herrera', and 'Paco Rabanne' have the highest average selling prices
2. These brands represent premium or luxury segments within the platform
3. High average prices indicate presence of high-value products
4. However, results may be influenced by low product counts for these brands

In [27]:
#5 brands with lowest average price

df.groupby('brand')['sale_price'].mean().sort_values().reset_index(name='avg_sale_price').head(5)

,brand,avg_sale_price
0,livon,3.0
1,narasus,10.0
2,dream bake,10.0
3,ala fresh,10.0
4,center fruit,10.0


##### # Insight:
1. The lowest average prices are observed for brands like 'Livon' and 'Dream Bake'
2. These values are likely driven by a small number of low-cost products such as sachets or entry-level items
3. This highlights a limitation of using average price without considering SKU distribution

In [28]:
# Top 5 Product Types by Average Selling Price

df.groupby('type')['sale_price'].mean().sort_values(ascending=False)\
.reset_index(name='avg_sale_price').head(5)

,type,avg_sale_price
0,Gas Stove,5718.250000
1,Cloth Dryer & Iron Table,2102.333333
2,Pressure Cookers,2003.500000
3,Eau De Parfum,1913.640876
4,Extra Virgin Olive Oil,1522.446078


##### Insight:
1. High average prices are observed for product types like 'Gas Stove' and 'Pressure Cookers', indicating premium durable goods
2. These product types represent high-value, low-frequency purchases compared to everyday consumables
3. The pricing pattern reflects a mix of essential grocery items and long-term household investments on the platform

In [29]:
#average rating by product type(top 5)

df.groupby('type')['rating'].mean().sort_values(ascending=False).reset_index(name='avg_rating').head(5)

,type,avg_rating
0,"Icetea, Non Aerated Drink",4.900000
1,Regular & White Vinegar,4.480000
2,Infant Formula,4.426667
3,"Wet Wipe, Pocket Tissues",4.425000
4,"Metal, Furniture Cleaner",4.425000


##### Insight:
1. Certain product types like 'Infant Formula' and 'Canola & Rapeseed Oil' show high average ratings, indicating strong customer satisfaction
2. These categories may represent trusted or high-quality products
3. However, high averages could be driven by a small number of rated items

In [30]:
# Calculate discount amount for each product
df['discount_amt'] = df['market_price']-df['sale_price']

In [31]:
#Which categories offer the highest average discount?

df.groupby('category')['discount_amt'].mean()\
.sort_values(ascending=False)\
.reset_index(name='avg_discount')

,category,avg_discount
0,"Kitchen, Garden & Pets",152.133039
1,Beauty & Hygiene,74.856105
2,Baby Care,61.807918
3,Gourmet & World Food,38.566874
4,"Foodgrains, Oil & Masala",36.964413
5,"Eggs, Meat & Fish",36.938000
6,Cleaning & Household,35.952603
7,Beverages,32.466437
8,"Bakery, Cakes & Dairy",15.078566
9,Fruits & Vegetables,13.544327


##### # Insight:
1. 'Kitchen, Garden & Pets' offers the highest average discount, followed by 'Beauty & Hygiene' and 'Baby Care'
2. These categories typically contain higher-priced products, resulting in larger absolute discount values
3. The variation in discount amounts highlights differences in pricing strategies across product categories

In [32]:
#What is the overall average discount on the platform?

df['discount_amt'].mean()

59.544726889993825

##### Insight:
1. The overall average discount on the platform is approximately ₹59.54
2. This indicates that, on average, products are discounted by around ₹60 from their market price

In [37]:
#Which products have extreme discount levels (>50%)?

df['discount_pct'] = (df['market_price'] - df['sale_price'])/df['market_price']*100

extreme_count = df[df['discount_pct']>50]
len(extreme_count[['product','type','discount_pct']])

437

##### Insight:
1. A significant number of products have discount levels above 50%, highlighting aggressive discounting practices
2. These extreme discounts are distributed across multiple product categories, suggesting platform-wide promotional strategies
3. However, such high discount percentages may also indicate inflated market prices or clearance of slow-moving inventory

In [34]:
# Which brands offer highest average discount percentage?
df.groupby('brand')['discount_pct'].mean().sort_values(ascending=False)\
.reset_index(name='highest_avg_discount').head(5)

,brand,highest_avg_discount
0,mud,80.000000
1,jensons,74.253281
2,mansaa,71.250764
3,triones,69.766040
4,mondsub,69.000000


#### Insight:
1. Extremely high average discount percentages are observed for brands such as 'Mud' and 'Jensons'
2. This suggests aggressive discounting or clearance strategies for certain products
3. However, these values may be skewed due to limited product counts per brand
4. A more reliable analysis would consider brands with a sufficient number of SKUs

In [35]:
df['product'] = df['product'].str.replace(r'[\r\n]+', ' ', regex=True)

In [36]:
df.to_csv('cleaned_df.csv', index=False)

### Key Insights & Conclusion :

1. The platform hosts 23,510 unique products across 11 categories, indicating a broad and diverse product assortment.

2. Beauty & Hygiene has the highest SKU count, indicating a highly competitive and fragmented category. This can make product discovery difficult for users, so improving filters and recommendation systems could enhance user experience.

3. Around 30% of products lack ratings, which can reduce customer trust and negatively impact conversion rates. Encouraging users to leave reviews through incentives or reminders could improve rating coverage.

4. A significant number of products are heavily discounted, which may indicate a strategy to drive sales volume. However, excessive discounting could impact margins, so analyzing the effectiveness of these discounts is important.

5. 'Beverages' has the highest average rating (~4.08), indicating strong customer satisfaction.

6. The platform includes 2,312 distinct brands, highlighting a diverse and competitive ecosystem. This may increase customer choice but also requires effective brand positioning and discovery mechanisms to help users navigate options.

7. Categories like Baby Care and Kitchen, Garden & Pets have the highest average prices, while Fruits & Vegetables has the lowest average price and no ratings. This suggests that users may be less likely to rate low-cost, frequently purchased items, whereas higher-value products receive more engagement. This behavior should be considered when evaluating product quality using ratings.